In [1]:
import requests
import pandas as pd
from io import StringIO

In [16]:
url = "https://www.waterqualitydata.us/data/Result/search"

# Note: using Result/search instead of Station/search
# This returns actual measurements not just station info
params = {
    "bBox": "-80.65,24.85,-80.20,25.35",
    "siteType": "Estuary",
    "startDateLo": "01-01-2015",
    "startDateHi": "12-31-2024",
    "sampleMedia": "Water",
    "characteristicName": [
        "Turbidity",
        "Salinity",
        "Temperature, water",
        "Chlorophyll a",
        "Depth, Secchi disk depth"
    ],
    "mimeType": "csv"
}

print("Downloading Upper Keys water quality results...")
response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    df = pd.read_csv(StringIO(response.text))
    print(f"Total rows: {df.shape[0]}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())
    
    # Save
    df.to_csv("upper_keys_results.csv", index=False)
    print("Saved as upper_keys_results.csv")
else:
    print(f"Error: {response.text[:500]}")

Status: 200
Total rows: 8056
Columns: ['OrganizationIdentifier', 'OrganizationFormalName', 'ActivityIdentifier', 'ActivityTypeCode', 'ActivityMediaName', 'ActivityMediaSubdivisionName', 'ActivityStartDate', 'ActivityStartTime/Time', 'ActivityStartTime/TimeZoneCode', 'ActivityEndDate', 'ActivityEndTime/Time', 'ActivityEndTime/TimeZoneCode', 'ActivityDepthHeightMeasure/MeasureValue', 'ActivityDepthHeightMeasure/MeasureUnitCode', 'ActivityDepthAltitudeReferencePointText', 'ActivityTopDepthHeightMeasure/MeasureValue', 'ActivityTopDepthHeightMeasure/MeasureUnitCode', 'ActivityBottomDepthHeightMeasure/MeasureValue', 'ActivityBottomDepthHeightMeasure/MeasureUnitCode', 'ProjectIdentifier', 'ActivityConductingOrganizationText', 'MonitoringLocationIdentifier', 'ActivityCommentText', 'SampleAquifer', 'HydrologicCondition', 'HydrologicEvent', 'SampleCollectionMethod/MethodIdentifier', 'SampleCollectionMethod/MethodIdentifierContext', 'SampleCollectionMethod/MethodName', 'SampleCollectionEquipmentN

In [36]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ---------------------------
# STEP 1 — download station metadata
# ---------------------------
station_url = (
    "https://www.waterqualitydata.us/data/Station/search?"
    "bBox=-80.65,24.90,-80.20,25.35&"
    "mimeType=csv"
)

stations = pd.read_csv(station_url)

# ---------------------------
# STEP 2 — keep ONLY needed columns
# ---------------------------
stations_clean = stations[[
    "MonitoringLocationIdentifier",
    "LatitudeMeasure",
    "LongitudeMeasure"
]].dropna().copy()

# rename for consistency
stations_clean = stations_clean.rename(columns={
    "LatitudeMeasure": "Latitude",
    "LongitudeMeasure": "Longitude"
})

# ensure one row per station (critical fix)
stations_clean = stations_clean.drop_duplicates("MonitoringLocationIdentifier")

# ---------------------------
# STEP 3 — clean df_pivot BEFORE merge
# ---------------------------
df_pivot = df_pivot.drop(
    columns=["Latitude", "Longitude", "MonitoringLocationIdentifier"],
    errors="ignore"
)

# ---------------------------
# STEP 4 — MERGE (ONLY ONCE)
# ---------------------------
df_pivot = df_pivot.merge(
    stations_clean,
    left_on="Station",
    right_on="MonitoringLocationIdentifier",
    how="left",
    validate="many_to_one"
)

# remove duplicate key column
df_pivot = df_pivot.drop(columns=["MonitoringLocationIdentifier"])

# ---------------------------
# STEP 5 — CREATE GEOMETRY
# ---------------------------
geometry = [
    Point(xy) for xy in zip(df_pivot["Longitude"], df_pivot["Latitude"])
]

wq_gdf = gpd.GeoDataFrame(
    df_pivot,
    geometry=geometry,
    crs="EPSG:4326"
)

# ---------------------------
# STEP 6 — sanity check
# ---------------------------
print(wq_gdf.head())
print("\nMissing coordinates:", wq_gdf["geometry"].isna().sum())

             Station                                      Organization  \
0  21FLDADE_WQX-BB47  Dade Environmental Resource Management (Florida)   
1  21FLDADE_WQX-BB50  Dade Environmental Resource Management (Florida)   
2  21FLDADE_WQX-BB51  Dade Environmental Resource Management (Florida)   
3  21FLSFWM_WQX-6598           South Florida Water Management District   
4  21FLSFWM_WQX-6599           South Florida Water Management District   

   Secchi_Depth   Salinity  Temperature  Turbidity  \
0           NaN  33.746667    25.330000        0.5   
1           NaN  30.970000    25.223333        0.8   
2           NaN  30.806667    25.396667        0.4   
3           2.5  35.000000    21.000000        1.8   
4           1.6  32.800000    21.500000        2.2   

  MonitoringLocationIdentifier_x  LatitudeMeasure_x  LongitudeMeasure_x  \
0              21FLDADE_WQX-BB47          25.336794          -80.320077   
1              21FLDADE_WQX-BB50          25.229898          -80.376777   
2    

In [37]:
display(wq_gdf.head())

,Station,Organization,Secchi_Depth,Salinity,Temperature,Turbidity,MonitoringLocationIdentifier_x,LatitudeMeasure_x,LongitudeMeasure_x,MonitoringLocationIdentifier_y,LatitudeMeasure_y,LongitudeMeasure_y,LatitudeMeasure,LongitudeMeasure,Latitude,Longitude,geometry
0,21FLDADE_WQX-BB47,Dade Environmental Resource Management (Florida),NaN,33.746667,25.330000,0.5,21FLDADE_WQX-BB47,25.336794,-80.320077,21FLDADE_WQX-BB47,25.336794,-80.320077,25.336794,-80.320077,25.336794,-80.320077,POINT (-80.32008 25.33679)
1,21FLDADE_WQX-BB50,Dade Environmental Resource Management (Florida),NaN,30.970000,25.223333,0.8,21FLDADE_WQX-BB50,25.229898,-80.376777,21FLDADE_WQX-BB50,25.229898,-80.376777,25.229898,-80.376777,25.229898,-80.376777,POINT (-80.37678 25.2299)
2,21FLDADE_WQX-BB51,Dade Environmental Resource Management (Florida),NaN,30.806667,25.396667,0.4,21FLDADE_WQX-BB51,25.251496,-80.414079,21FLDADE_WQX-BB51,25.251496,-80.414079,25.251496,-80.414079,25.251496,-80.414079,POINT (-80.41408 25.2515)
3,21FLSFWM_WQX-6598,South Florida Water Management District,2.5,35.000000,21.000000,1.8,21FLSFWM_WQX-6598,25.174050,-80.423081,21FLSFWM_WQX-6598,25.174050,-80.423081,25.174050,-80.423081,25.174050,-80.423081,POINT (-80.42308 25.17405)
4,21FLSFWM_WQX-6599,South Florida Water Management District,1.6,32.800000,21.500000,2.2,21FLSFWM_WQX-6599,25.206681,-80.440400,21FLSFWM_WQX-6599,25.206681,-80.440400,25.206681,-80.440400,25.206681,-80.440400,POINT (-80.4404 25.20668)


In [17]:
# Check all parameters and their counts
print("Parameters in dataset:")
print(df["CharacteristicName"].value_counts())

print(f"\nTotal unique parameters: {df['CharacteristicName'].nunique()}")

Parameters in dataset:
CharacteristicName
Salinity                    2696
Temperature, water          2693
Turbidity                   1574
Depth, Secchi disk depth    1093
Name: count, dtype: int64

Total unique parameters: 4


In [18]:
# Check if chlorophyll-a exists under a different name
all_params = df["CharacteristicName"].unique()
print(sorted(all_params))

['Depth, Secchi disk depth', 'Salinity', 'Temperature, water', 'Turbidity']


In [ ]:
# Step 1 — keep relevant columns
cols_to_keep = [
    "OrganizationFormalName",
    "MonitoringLocationIdentifier",
    "ActivityStartDate",
    "ActivityStartTime/Time",
    "ActivityDepthHeightMeasure/MeasureValue",
    "CharacteristicName",
    "ResultMeasureValue",
    "ResultMeasure/MeasureUnitCode",
    "MeasureQualifierCode",
    "ResultStatusIdentifier"
]
df_clean = df[cols_to_keep].copy()

# Step 2 — rename columns
df_clean = df_clean.rename(columns={
    "OrganizationFormalName": "Organization",
    "MonitoringLocationIdentifier": "Station",
    "ActivityStartDate": "Date",
    "ActivityStartTime/Time": "Time",
    "ActivityDepthHeightMeasure/MeasureValue": "Depth",
    "CharacteristicName": "Parameter",
    "ResultMeasureValue": "Value",
    "ResultMeasure/MeasureUnitCode": "Units",
    "MeasureQualifierCode": "Qualifier",
    "ResultStatusIdentifier": "Status"
})

# Step 3 — combine date and time into datetime
df_clean["Datetime"] = pd.to_datetime(
    df_clean["Date"] + " " + df_clean["Time"].fillna("00:00:00"),
    errors="coerce"
)

# Step 4 — filter to 2015 onwards
df_clean = df_clean[df_clean["Datetime"] >= "2015-01-01"]
print(f"After date filter: {df_clean.shape[0]} rows")

# Step 5 — check and filter status
print("\nStatus values:")
print(df_clean["Status"].value_counts())
df_clean = df_clean[
    df_clean["Status"].isin(["Accepted", "Final", "Validated", "Historical"])
]
print(f"After status filter: {df_clean.shape[0]} rows")

# Step 6 — filter to four parameters
params_of_interest = [
    "Turbidity",
    "Salinity",
    "Temperature, water",
    "Depth, Secchi disk depth"
]
df_clean = df_clean[df_clean["Parameter"].isin(params_of_interest)]
print(f"After parameter filter: {df_clean.shape[0]} rows")

# Step 7 — convert value to numeric
df_clean["Value"] = pd.to_numeric(df_clean["Value"], errors="coerce")

# Step 8 — drop rows with no measurement
df_clean = df_clean.dropna(subset=["Value"])
print(f"After dropping NaN values: {df_clean.shape[0]} rows")

# Step 9 — extract date only
df_clean["Date"] = df_clean["Datetime"].dt.date

# Step 10 — pivot parameters into columns
df_pivot = df_clean.pivot_table(
    index=["Date", "Station", "Organization"],
    columns="Parameter",
    values="Value",
    aggfunc="mean"
).reset_index()

df_pivot.columns.name = None

# Step 11 — rename parameter columns
df_pivot = df_pivot.rename(columns={
    "Temperature, water": "Temperature",
    "Depth, Secchi disk depth": "Secchi_Depth"
})

# Step 12 — set Date as index and sort
df_pivot = df_pivot.set_index("Date").sort_index()

# Step 13 — final summary
print(f"\nFinal dataframe shape: {df_pivot.shape}")
print(f"Date range: {df_pivot.index.min()} to {df_pivot.index.max()}")
print(f"Unique stations: {df_pivot['Station'].nunique()}")
print(f"\nNaN counts per parameter:")
print(df_pivot[["Turbidity", "Salinity",
                 "Temperature", "Secchi_Depth"]].isnull().sum())
print(f"\nPreview:")
print(df_pivot.head(10))

# Step 14 — save
df_pivot.to_csv("upper_keys_cleaned.csv")
print("\nSaved as upper_keys_cleaned.csv")

After date filter: 8056 rows

Status values:
Status
Final    8056
Name: count, dtype: int64
After status filter: 8056 rows
After parameter filter: 8056 rows
After dropping NaN values: 8045 rows

Final dataframe shape: (1738, 6)
Date range: 2015-01-07 to 2024-12-11
Unique stations: 56

NaN counts per parameter:
Turbidity       171
Salinity        157
Temperature     135
Secchi_Depth    656
dtype: int64

Preview:
                      Station  \
Date                            
2015-01-07  21FLDADE_WQX-BB47   
2015-01-07  21FLDADE_WQX-BB50   
2015-01-07  21FLDADE_WQX-BB51   
2015-02-03  21FLSFWM_WQX-6598   
2015-02-03  21FLSFWM_WQX-6599   
2015-02-03  21FLSFWM_WQX-6600   
2015-02-03  21FLSFWM_WQX-6601   
2015-02-03  21FLSFWM_WQX-6602   
2015-02-03  21FLSFWM_WQX-6603   
2015-02-03  21FLSFWM_WQX-6604   

                                                Organization  Secchi_Depth  \
Date                                                                         
2015-01-07  Dade Environmental R

In [38]:
display(df_pivot.head())

,Station,Organization,Secchi_Depth,Salinity,Temperature,Turbidity,MonitoringLocationIdentifier_x,LatitudeMeasure_x,LongitudeMeasure_x,MonitoringLocationIdentifier_y,LatitudeMeasure_y,LongitudeMeasure_y,LatitudeMeasure,LongitudeMeasure,Latitude,Longitude
0,21FLDADE_WQX-BB47,Dade Environmental Resource Management (Florida),NaN,33.746667,25.330000,0.5,21FLDADE_WQX-BB47,25.336794,-80.320077,21FLDADE_WQX-BB47,25.336794,-80.320077,25.336794,-80.320077,25.336794,-80.320077
1,21FLDADE_WQX-BB50,Dade Environmental Resource Management (Florida),NaN,30.970000,25.223333,0.8,21FLDADE_WQX-BB50,25.229898,-80.376777,21FLDADE_WQX-BB50,25.229898,-80.376777,25.229898,-80.376777,25.229898,-80.376777
2,21FLDADE_WQX-BB51,Dade Environmental Resource Management (Florida),NaN,30.806667,25.396667,0.4,21FLDADE_WQX-BB51,25.251496,-80.414079,21FLDADE_WQX-BB51,25.251496,-80.414079,25.251496,-80.414079,25.251496,-80.414079
3,21FLSFWM_WQX-6598,South Florida Water Management District,2.5,35.000000,21.000000,1.8,21FLSFWM_WQX-6598,25.174050,-80.423081,21FLSFWM_WQX-6598,25.174050,-80.423081,25.174050,-80.423081,25.174050,-80.423081
4,21FLSFWM_WQX-6599,South Florida Water Management District,1.6,32.800000,21.500000,2.2,21FLSFWM_WQX-6599,25.206681,-80.440400,21FLSFWM_WQX-6599,25.206681,-80.440400,25.206681,-80.440400,25.206681,-80.440400


In [24]:
import geopandas as gpd
import rasterio as rio
import rioxarray
# Load reef polygon or study area
reef_shape = gpd.read_file(r"C:\Users\ianra\Downloads\Python Lessons\FLUpperKeys\UpperKeys_FL.shp")

In [39]:
from shapely.geometry import Point

# Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(df_pivot['Longitude'], df_pivot['Latitude'])]
wq_gdf = gpd.GeoDataFrame(df_pivot, geometry=geometry)

# Set CRS (must match your shapefile CRS, here assuming UTM after reprojection)
wq_gdf = wq_gdf.set_crs("EPSG:32617")


In [40]:
display(wq_gdf.head())

,Station,Organization,Secchi_Depth,Salinity,Temperature,Turbidity,MonitoringLocationIdentifier_x,LatitudeMeasure_x,LongitudeMeasure_x,MonitoringLocationIdentifier_y,LatitudeMeasure_y,LongitudeMeasure_y,LatitudeMeasure,LongitudeMeasure,Latitude,Longitude,geometry
0,21FLDADE_WQX-BB47,Dade Environmental Resource Management (Florida),NaN,33.746667,25.330000,0.5,21FLDADE_WQX-BB47,25.336794,-80.320077,21FLDADE_WQX-BB47,25.336794,-80.320077,25.336794,-80.320077,25.336794,-80.320077,POINT (-80.32 25.337)
1,21FLDADE_WQX-BB50,Dade Environmental Resource Management (Florida),NaN,30.970000,25.223333,0.8,21FLDADE_WQX-BB50,25.229898,-80.376777,21FLDADE_WQX-BB50,25.229898,-80.376777,25.229898,-80.376777,25.229898,-80.376777,POINT (-80.377 25.23)
2,21FLDADE_WQX-BB51,Dade Environmental Resource Management (Florida),NaN,30.806667,25.396667,0.4,21FLDADE_WQX-BB51,25.251496,-80.414079,21FLDADE_WQX-BB51,25.251496,-80.414079,25.251496,-80.414079,25.251496,-80.414079,POINT (-80.414 25.251)
3,21FLSFWM_WQX-6598,South Florida Water Management District,2.5,35.000000,21.000000,1.8,21FLSFWM_WQX-6598,25.174050,-80.423081,21FLSFWM_WQX-6598,25.174050,-80.423081,25.174050,-80.423081,25.174050,-80.423081,POINT (-80.423 25.174)
4,21FLSFWM_WQX-6599,South Florida Water Management District,1.6,32.800000,21.500000,2.2,21FLSFWM_WQX-6599,25.206681,-80.440400,21FLSFWM_WQX-6599,25.206681,-80.440400,25.206681,-80.440400,25.206681,-80.440400,POINT (-80.44 25.207)
